#### Load Libraries

In [1]:
import duckdb
import pandas as pd
import os
from dotenv import load_dotenv
import anthropic
from concurrent.futures import ThreadPoolExecutor, as_completed
import unicodedata
import re
import random
import time
import warnings
warnings.filterwarnings('ignore')

#### Set up Anthropic Client

In [2]:
load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
print("Client initialized successfully")

Client initialized successfully


#### Setup DuckDB connection

In [3]:
conn = duckdb.connect('/app/data/analytics.duckdb')

#### Source Data

In [4]:
# Load all review data into a dataframe
df = conn.execute("""
    SELECT *
    FROM raw.order_reviews
""").df()

print("Shape -", df.shape)
df.head()

Shape - (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [5]:
# Portugese-English translations for small words like Good, Great, Bad.
lookup_df = pd.read_csv('/app/short_review_lookup.csv')
short_review_lookup = dict(zip(lookup_df['portuguese'], lookup_df['english']))

### Translations

#### Short 1-2 word translations are done using a translation dictionary

In [6]:
# Separate into lookup vs API translation buckets
df['message_length'] = df['review_comment_message'].str.len()
df_with_message = df[df['review_comment_message'].notna()].copy()

df_short = df_with_message[df_with_message['message_length'] < 10]
df_translate = df_with_message[df_with_message['message_length'] >= 10]

print(f"Total reviews with messages: {len(df_with_message):,}")
print(f"Short reviews (lookup): {len(df_short):,}")
print(f"Translatable reviews (API): {len(df_translate):,}")
print(f"Dropped (no message): {df['review_comment_message'].isna().sum():,}")

Total reviews with messages: 40,977
Short reviews (lookup): 2,954
Translatable reviews (API): 38,023
Dropped (no message): 58,247


#### Normalization
Normalizing text by removing accents and special characters to enable consistent lookup against our translation dictionary

In [7]:
def normalize(text):
    text = text.lower().strip()
    # Remove accents
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    # Remove special chars
    text = re.sub(r'[^a-z ]', '', text)
    # Collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Re-apply with updated normalize
df_short['normalized_message'] = df_short['review_comment_message'].apply(normalize)
df_short['translated_message'] = df_short['normalized_message'].map(short_review_lookup)
df_short_translated = df_short[df_short['translated_message'].notna()].copy()

print(f"Successfully mapped: {len(df_short_translated):,}")
print(f"Dropped (unmapped/noise): {df_short['translated_message'].isna().sum():,}")

Successfully mapped: 2,507
Dropped (unmapped/noise): 447


### Function to Translate using Claude Haiku Model

In [8]:
def translate_batch(reviews, client):
    numbered = "\n".join([f"{i+1}. {text}" for i, text in enumerate(reviews)])
    
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=4096,
        messages=[{
            "role": "user",
            "content": f"""Translate the following Portuguese reviews to English. 
                        Return exactly {len(reviews)} translations, one per line, numbered 1 to {len(reviews)}.
                        Do not add any extra lines, commentary, or blank lines between translations.
                        {numbered}"""
        }]
    )
    
    lines = message.content[0].text.strip().split('\n')
    translations = [re.sub(r'^\d+\.\s*', '', line).strip() 
                   for line in lines 
                   if re.match(r'^\d+\.', line.strip())]
    
    return translations

#### Test on 5 reviews first

In [9]:
sample = df_translate['review_comment_message'].head(5).tolist()
result = translate_batch(sample, client)

for original, translated in zip(sample, result):
    print(f"Original:   {original}")
    print(f"Translated: {translated}")
    print()

Original:   Recebi bem antes do prazo estipulado.
Translated: I received it well before the stipulated deadline.

Original:   Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa
Translated: Congratulations Lannister stores I loved shopping on the Internet safe and practical Congratulations to everyone happy Easter

Original:   aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho
Translated: Efficient device. On the website the device brand is printed as 3disinfector and upon arrival it has another name...update with the correct brand since it is the same device

Original:   Mas um pouco ,travando...pelo valor ta Boa.

Translated: A little slow, freezing...for the price it's good.

Original:   Vendedor confiável, produto ok e entrega antes do prazo.
Translated: Reliable seller, product ok and delivery before the deadli

### Batch Translation functions

In [10]:
def translate_batch_with_index(args):
    idx, batch, client = args
    max_retries = 3
    
    for attempt in range(max_retries):
        try:
            result = translate_batch(batch, client)
            if len(result) != len(batch):
                result = result + [None] * (len(batch) - len(result))
            return idx, result
        except Exception as e:
            if '429' in str(e) and attempt < max_retries - 1:
                wait = 2 ** attempt + random.uniform(0, 1)
                print(f"Rate limited on batch {idx}, retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"Batch {idx} failed after {max_retries} attempts: {e}")
                return idx, [None] * len(batch)

def translate_all_reviews_parallel(df, client, batch_size=100, max_workers=5, column='review_comment_message'):
    batches = []
    for i in range(0, len(df), batch_size):
        batch = df[column].iloc[i:i+batch_size].tolist()
        batches.append((i // batch_size, batch, client))
    
    results = {}
    total = len(batches)
    completed = 0
    start = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(translate_batch_with_index, b): b[0] for b in batches}
        
        for future in as_completed(futures):
            idx, translations = future.result()
            results[idx] = translations
            completed += 1
            
            elapsed = time.time() - start
            reviews_done = completed * batch_size
            pct = (completed / total) * 100
            eta = (total - completed) * (elapsed / completed) if completed > 0 else 0
            
            print(f"[{pct:.1f}%] Batch {completed}/{total} | "
                  f"~{min(reviews_done, len(df)):,} reviews | "
                  f"Elapsed: {elapsed/60:.1f}m | "
                  f"ETA: {eta/60:.1f}m")
    
    all_translations = []
    for i in range(total):
        all_translations.extend(results[i])
    
    return all_translations

#### Test on 200 Reviews

In [11]:
start = time.time()
test_df = df_translate.head(200)
test_translations = translate_all_reviews_parallel(test_df, client, batch_size=100, max_workers=5)
elapsed = time.time() - start

print(f"\nDone. Translated {len([t for t in test_translations if t is not None])} of 200 reviews")
print(f"Time elapsed: {elapsed:.1f} seconds")
print(f"Estimated full run time: {(elapsed / 200) * len(df_translate) / 60:.1f} minutes")

[50.0%] Batch 1/2 | ~100 reviews | Elapsed: 0.5m | ETA: 0.5m
[100.0%] Batch 2/2 | ~200 reviews | Elapsed: 0.5m | ETA: 0.0m

Done. Translated 200 of 200 reviews
Time elapsed: 30.6 seconds
Estimated full run time: 96.9 minutes


In [12]:
BATCH_SIZE = 100
print(f"Estimated batches: {len(df_translate) // BATCH_SIZE + 1}")
print(f"Estimated time at 0.5s/batch: ~{(len(df_translate) // BATCH_SIZE + 1) * elapsed / 60:.1f} minutes\n")

Estimated batches: 381
Estimated time at 0.5s/batch: ~194.1 minutes



<br>

**To avoid blocking development on a 3+ hour translation job, we work with a stratified 5,000 review sample while submitting the full 38K to Anthropic's Batch API (Check `02_submit_batch.ipynb`) for asynchronous processing at 50% cost.**

#### Sample Data Set Creation

In [13]:
# Stratified sample - 1000 reviews per score
sample_df = df_translate.groupby('review_score', group_keys=False).apply(
    lambda x: x.sample(min(1000, len(x)), random_state=42)
).reset_index(drop=True)

print(f"Sample size: {len(sample_df)}")
print(sample_df['review_score'].value_counts().sort_index())

Sample size: 5000
review_score
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64


#### Review Comment Message Transalations

In [14]:
start = time.time()

sample_translations = translate_all_reviews_parallel(sample_df, client, batch_size=100, max_workers=5)

elapsed = time.time() - start
print(f"\nDone. Translated {len([t for t in sample_translations if t is not None]):,} of {len(sample_df):,} reviews")
print(f"Total time: {elapsed/60:.1f} minutes")

[2.0%] Batch 1/50 | ~100 reviews | Elapsed: 0.7m | ETA: 32.0m
[4.0%] Batch 2/50 | ~200 reviews | Elapsed: 0.7m | ETA: 16.0m
[6.0%] Batch 3/50 | ~300 reviews | Elapsed: 0.7m | ETA: 10.5m
[8.0%] Batch 4/50 | ~400 reviews | Elapsed: 0.7m | ETA: 7.8m
[10.0%] Batch 5/50 | ~500 reviews | Elapsed: 0.7m | ETA: 6.7m
[12.0%] Batch 6/50 | ~600 reviews | Elapsed: 1.3m | ETA: 9.6m
[14.0%] Batch 7/50 | ~700 reviews | Elapsed: 1.3m | ETA: 8.1m
[16.0%] Batch 8/50 | ~800 reviews | Elapsed: 1.3m | ETA: 6.9m
[18.0%] Batch 9/50 | ~900 reviews | Elapsed: 1.4m | ETA: 6.3m
[20.0%] Batch 10/50 | ~1,000 reviews | Elapsed: 1.4m | ETA: 5.7m
[22.0%] Batch 11/50 | ~1,100 reviews | Elapsed: 2.6m | ETA: 9.1m
[24.0%] Batch 12/50 | ~1,200 reviews | Elapsed: 2.6m | ETA: 8.2m
[26.0%] Batch 13/50 | ~1,300 reviews | Elapsed: 2.6m | ETA: 7.4m
[28.0%] Batch 14/50 | ~1,400 reviews | Elapsed: 2.7m | ETA: 6.9m
[30.0%] Batch 15/50 | ~1,500 reviews | Elapsed: 2.7m | ETA: 6.3m
[32.0%] Batch 16/50 | ~1,600 reviews | Elapsed: 3.9m 

#### Join back the translated messages

In [15]:
sample_df['translated_message'] = sample_translations

# Check for any nulls
print(f"Successful translations: {sample_df['translated_message'].notna().sum():,}")
print(f"Failed translations: {sample_df['translated_message'].isna().sum():,}")

sample_df.head(n=3)

Successful translations: 5,000
Failed translations: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,message_length,translated_message
0,1856e1f2326a0ce3da86525e55b5c056,3bc25388dde67d943ffaf8db488cebc0,1,None,Comprei dois produtos e recebi um. Insatisfeit...,2017-07-06,2017-07-07 00:13:23,90.0,I bought two products and received one. Dissat...
1,ac7ebe51f59d79b99898c9d541b3868d,907424cd045f2dcb775507e367f9ab60,1,Demora no recebimento,Estou a 3 dias em casa sem sair aguardando o p...,2018-08-16,2018-08-16 12:22:19,70.0,I have been at home for 3 days without leaving...
2,44f7482ffa6b98bc55f68fd6452cb308,c130858ff611f05ccbfbac337cd09cdc,1,FABRICAÇÃO,"Origem do produto china, qualidade ruim, na ho...",2018-07-01,2018-07-22 12:31:18,147.0,"Product origin China, poor quality, at the tim..."


In [16]:
print(f"Reviews with titles: {sample_df['review_comment_title'].notna().sum():,}")
print(f"Reviews without titles: {sample_df['review_comment_title'].isna().sum():,}")

Reviews with titles: 1,152
Reviews without titles: 3,848


#### Review Comment Title Transalations¶

In [18]:
sample_with_titles = sample_df[sample_df['review_comment_title'].notna()].copy()
print(f"Translating {len(sample_with_titles):,} titles...")

Translating 1,152 titles...


In [19]:
start = time.time()
title_translations = translate_all_reviews_parallel(
    sample_with_titles,
    client,
    batch_size=100,
    max_workers=5,
    column='review_comment_title'
)
elapsed = time.time() - start

print(f"\nDone. Translated {len([t for t in title_translations if t is not None]):,} titles")
print(f"Time elapsed: {elapsed:.1f} seconds")

[8.3%] Batch 1/12 | ~100 reviews | Elapsed: 0.1m | ETA: 0.7m
[16.7%] Batch 2/12 | ~200 reviews | Elapsed: 0.1m | ETA: 0.3m
[25.0%] Batch 3/12 | ~300 reviews | Elapsed: 0.1m | ETA: 0.2m
[33.3%] Batch 4/12 | ~400 reviews | Elapsed: 0.1m | ETA: 0.1m
[41.7%] Batch 5/12 | ~500 reviews | Elapsed: 0.1m | ETA: 0.1m
[50.0%] Batch 6/12 | ~600 reviews | Elapsed: 0.1m | ETA: 0.1m
[58.3%] Batch 7/12 | ~700 reviews | Elapsed: 0.1m | ETA: 0.1m
[66.7%] Batch 8/12 | ~800 reviews | Elapsed: 0.1m | ETA: 0.1m
[75.0%] Batch 9/12 | ~900 reviews | Elapsed: 0.1m | ETA: 0.0m
[83.3%] Batch 10/12 | ~1,000 reviews | Elapsed: 0.1m | ETA: 0.0m
[91.7%] Batch 11/12 | ~1,100 reviews | Elapsed: 0.2m | ETA: 0.0m
[100.0%] Batch 12/12 | ~1,152 reviews | Elapsed: 0.2m | ETA: 0.0m

Done. Translated 1,152 titles
Time elapsed: 11.5 seconds


In [20]:
# Attach title translations
sample_with_titles['translated_title'] = title_translations

# Merge back into sample_df
sample_df = sample_df.merge(
    sample_with_titles[['review_id', 'translated_title']],
    on='review_id',
    how='left'
)

print(f"Translated titles: {sample_df['translated_title'].notna().sum():,}")
print(f"No title: {sample_df['translated_title'].isna().sum():,}")

sample_df.head()

Translated titles: 1,156
No title: 3,848


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,message_length,translated_message,translated_title
0,1856e1f2326a0ce3da86525e55b5c056,3bc25388dde67d943ffaf8db488cebc0,1,None,Comprei dois produtos e recebi um. Insatisfeit...,2017-07-06,2017-07-07 00:13:23,90.0,I bought two products and received one. Dissat...,NaN
1,ac7ebe51f59d79b99898c9d541b3868d,907424cd045f2dcb775507e367f9ab60,1,Demora no recebimento,Estou a 3 dias em casa sem sair aguardando o p...,2018-08-16,2018-08-16 12:22:19,70.0,I have been at home for 3 days without leaving...,Delay in receipt
2,44f7482ffa6b98bc55f68fd6452cb308,c130858ff611f05ccbfbac337cd09cdc,1,FABRICAÇÃO,"Origem do produto china, qualidade ruim, na ho...",2018-07-01,2018-07-22 12:31:18,147.0,"Product origin China, poor quality, at the tim...",MANUFACTURING
3,538cd4be7d4ee6b34f716c60f8c6f718,8932f7d67eec4b3ed9a4d5ae0b30e06a,1,None,AGUARDO UM RETORNO PARA RESOLVERMOS O ASSUNTO ...,2017-08-02,2017-08-03 22:32:35,70.0,I AWAIT A RESPONSE TO RESOLVE THE MATTER AND M...,NaN
4,a492b3efc47eb9096c41de9d772cc4f3,b985389048e8bbb41d4993f4bd86f47f,1,None,Ainda não recebi,2017-09-22,2017-09-25 22:08:27,16.0,I still haven't received it.,NaN


#### Save as CSV

In [21]:
sample_df.to_csv('/app/data/translated_reviews_sample.csv', index=False)
print(f"Saved {len(sample_df):,} reviews to /app/data/translated_reviews_sample.csv")

Saved 5,004 reviews to /app/data/translated_reviews_sample.csv


#### Save to DuckDB

In [22]:
conn.execute("CREATE SCHEMA IF NOT EXISTS llm_outputs")
conn.execute("DROP TABLE IF EXISTS llm_outputs.translated_reviews_sample")
conn.execute("CREATE TABLE llm_outputs.translated_reviews_sample AS SELECT * FROM sample_df")

print(f"Saved {len(sample_df):,} rows to llm_outputs.translated_reviews_sample")
print(conn.execute("SELECT COUNT(*) FROM llm_outputs.translated_reviews_sample").fetchone())

Saved 5,004 rows to llm_outputs.translated_reviews_sample
(5004,)
